In [ ]:
#1. initialization
import arcpy
import os
import glob

# Check out the Spatial Analyst extension
arcpy.CheckOutExtension("Spatial")
arcpy.env.overwriteOutput = True

# Define your workspaces
ERC_DIR = os.path.join(base_dir, "data", "erc")
gdb     = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
arcpy.env.workspace = ERC_DIR

# 1. Gather all  downloaded GRIDMET ERC NetCDF files
nc_files = glob.glob(os.path.join(ERC_DIR, "*.nc"))
print(f"Found {len(nc_files)} NetCDF climate files to process.")

# 2. Point to your fire features dataset (Update this path to actual fire shapefile/gdb)
fire_features = os.path.join(gdb, "SEFM_events_94_24_matching_complete")

In [ ]:
# 2. Define where the brand-new points layer should be saved
gdb         = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
fire_points = os.path.join(gdb, "SEFM_events_94_24_centroids")

print("Converting polygons to centroids...")
arcpy.management.FeatureToPoint(
    in_features=fire_features,     # This looks at master polygon path
    out_feature_class=fire_points, # This is the new points layer it will create
    point_location="INSIDE" 
)
print("Centroid point layer created successfully.")

In [ ]:
# 3. Define where the output sample table will be saved in your GDB
import xarray as xr

# Define where the output sample table will be saved in your GDB
gdb                  = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
sampled_output_table = os.path.join(gdb, "fire_grid_samples")

# 1. Inspect the NetCDF file to see its true dimension/variable names
template_nc = nc_files[0]
print(f"Reading file structure for: {os.path.basename(template_nc)}")

with xr.open_dataset(template_nc) as ds:
    # Get the key dimension names (usually lon/lat or lon/lat but let's confirm)
    dimensions = list(ds.dims)
    # Find the target variable name (looking for something with erc or fuel moisture)
    variables = [v for v in ds.variables if v not in dimensions]
    
    print(f"-> Found Dimensions in file: {dimensions}")
    print(f"-> Found Variables in file: {variables}\n")

    # Map them automatically based on standard GRIDMET structures
    x_dim = 'lon' if 'lon' in dimensions else ('lon' if 'lon' in dimensions else dimensions[0])
    y_dim = 'lat' if 'lat' in dimensions else ('lat' if 'lat' in dimensions else dimensions[1])
    time_dim = 'day' if 'day' in dimensions else [d for d in dimensions if d not in [x_dim, y_dim]][0]
    
    # Target ERC variable (usually 'erc' in standard GRIDMET)
    var_name = 'erc' if 'erc' in variables else variables[0]

print(f"Using Auto-detected parameters -> Var: '{var_name}', X: '{x_dim}', Y: '{y_dim}', Band: '{time_dim}'")

# 2. Build the temporary raster layer using the detected names
temp_raster_layer = "erc_template_raster"
print("Creating temporary spatial template from NetCDF...")
arcpy.md.MakeNetCDFRasterLayer(
    in_netCDF_file=template_nc,
    variable=var_name,
    x_dimension=x_dim,
    y_dimension=y_dim,
    out_raster_layer=temp_raster_layer,
    band_dimension=time_dim
)

# 3. Run the Sample tool
print("Sampling grid cell locations for all centroids. This may take a moment.")
arcpy.sa.Sample(
    in_rasters=[temp_raster_layer],
    in_location_data=fire_points,      
    out_table=sampled_output_table,
    resampling_type="NEAREST"
)

print("Sampling complete. 'fire_grid_samples' table has been added to Geodatabase.")

In [ ]:
#4 inspect sample table
import arcpy
import pandas as pd

# Path to the sample table just created in GDB
gdb               = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
sample_table_path = os.path.join(gdb, "fire_grid_samples")

# Print all field names so can see exactly what the Sample tool generated
fields = [f.name for f in arcpy.ListFields(sample_table_path)]
print("--- Fields found in your fire_grid_samples table ---")
for field in fields:
    print(f"- {field}")

# Let's read just the first 3 rows to see what the data looks like
print("\n--- First 3 rows of data ---")
cursor = arcpy.da.SearchCursor(sample_table_path, ["*"])
df_preview = pd.DataFrame(list(cursor), columns=fields).head(3)
print(df_preview)

In [ ]:
#5 ERC time series extraction and percentile calculation. This took about 10 hours to run
import arcpy
import xarray as xr
import pandas as pd
import numpy as np
import os
import sys

# --- PATH CONFIGURATION ---
ERC_DIR           = os.path.join(base_dir, "erc")
gdb               = os.path.join(base_dir, "output", "ClassiFIRE.gdb")
SAMPLE_TABLE      = os.path.join(gdb, "fire_grid_samples")
FIRE_POINTS       = os.path.join(gdb, "SEFM_events_94_24_centroids")
ORIGINAL_POLYGONS = os.path.join(gdb, "SEFM_events_94_24_matching_complete")
OUTPUT_CSV        = os.path.join(base_dir, "data", "fire_erc_percentiles.csv")

print("1. Loading spatial grid samples from GDB table...")
sample_fields = ["SEFM_events_94_24_centroids"]
with arcpy.da.SearchCursor(SAMPLE_TABLE, sample_fields) as cursor:
    df_spatial = pd.DataFrame(list(cursor), columns=sample_fields)

print("2. Extracting true Lat/Lon coordinates from the original centroid features...")
# Fixed the index from row[2] to row[1] to properly unpack the SHAPE@XY tuple
points_geo = []
with arcpy.da.SearchCursor(FIRE_POINTS, ["OBJECTID", "SHAPE@XY"], spatial_reference=arcpy.SpatialReference(4326)) as cursor:
    for row in cursor:
        points_geo.append({'centroid_id': row[0], 'lon_deg': row[1][0], 'lat_deg': row[1][1]})
df_geo = pd.DataFrame(points_geo)

print("3. Loading master polygon attributes (event_id & midpoint_date)...")
poly_fields = ["OBJECTID", "event_id", "midpoint_date"] 
with arcpy.da.SearchCursor(ORIGINAL_POLYGONS, poly_fields) as cursor:
    df_poly_attrs = pd.DataFrame(list(cursor), columns=poly_fields)

print("4. Linking tables and coordinates together in memory...")
# First attach the master attributes
df_fires = pd.merge(df_spatial, df_poly_attrs, left_on="SEFM_events_94_24_centroids", right_on="OBJECTID")
# Next attach the true Lat/Lon degrees we extracted
df_fires = pd.merge(df_fires, df_geo, left_on="SEFM_events_94_24_centroids", right_on="centroid_id")

df_fires['midpoint_date'] = pd.to_datetime(df_fires['midpoint_date'])
df_fires = df_fires.dropna(subset=['midpoint_date', 'event_id']).copy()
print(f"-> Successfully synchronized {len(df_fires)} fire records with real coordinates.")

print("\n5. Lazy-loading NetCDF files...")
ds = xr.open_mfdataset(os.path.join(ERC_DIR, "*.nc"), combine="by_coords")
erc_cube = ds['energy_release_component-g']

lon_coord = 'lon' if 'lon' in ds.coords else [c for c in ds.coords if 'lon' in c.lower()][0]
lat_coord = 'lat' if 'lat' in ds.coords else [c for c in ds.coords if 'lat' in c.lower()][0]
time_coord = 'day' if 'day' in ds.coords else 'time'

netcdf_lons = ds[lon_coord].values
netcdf_lats = ds[lat_coord].values

print("\n6. Compressing dataset into unique grid cells...")
# Match the clean Lat/Lon degrees directly to the nearest NetCDF pixel grid coordinates
df_fires['target_lon'] = df_fires['lon_deg'].apply(lambda x: netcdf_lons[np.abs(netcdf_lons - x).argmin()])
df_fires['target_lat'] = df_fires['lat_deg'].apply(lambda y: netcdf_lats[np.abs(netcdf_lats - y).argmin()])

grouped = df_fires.groupby(['target_lon', 'target_lat'])
total_groups = len(grouped)
print(f"-> Compressed 2.2M fires into {total_groups} unique geographic grid cells.")

results = []
location_count = 0

print("\nProcessing locations (Live real-time status updates)...")

for (lon_val, lat_val), group in grouped:
    location_count += 1
    if location_count % 500 == 0 or location_count == total_groups:
        sys.stdout.write(f"\rProcessed {location_count} / {total_groups} grid locations...")
        sys.stdout.flush()
        
    try:
        # Pull the 45-year continuous history for this pixel using actual mapped coords
        pixel_history = erc_cube.sel({lon_coord: lon_val, lat_coord: lat_val}, method='nearest').values
        
        if np.isnan(pixel_history).all():
            continue
            
        time_index = ds[time_coord].values
        ts = pd.Series(pixel_history, index=pd.to_datetime(time_index))
        
        # Hyper-speed vectorized rolling window for all 45 years at once
        rolling_history = ts.rolling(window=31, center=True, min_periods=1).mean()
        
    except Exception as e:
        continue

    # Process all fires belonging to this exact cell via memory lookups
    for idx, row in group.iterrows():
        target_date = row['midpoint_date']
        
        if target_date in rolling_history.index:
            fire_absolute_mean = rolling_history.loc[target_date]
            
            if np.isnan(fire_absolute_mean):
                continue
                
            # Compute percentile rank against background climate totality (Option 2)
            percentile = np.mean(pixel_history < fire_absolute_mean) * 100.0
            
            results.append({
                'event_id': row['event_id'],
                'absolute_mean_erc': fire_absolute_mean,
                'totality_percentile': percentile
            })

print("\n\n7. Exporting compiled features to CSV...")
df_results = pd.DataFrame(results)
df_results.to_csv(OUTPUT_CSV, index=False)
print(f"SUCCESS. Continuous Random Forest features saved to: {OUTPUT_CSV}")